In [9]:
!pip install flask-ngrok==0.0.25 flask soundfile pydub librosa numpy tensorflow==2.20.0
!apt-get install ffmpeg -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import tensorflow as tf
from tensorflow.keras.models import load_model

# Auto-search for best_model.h5 anywhere inside MyDrive
search_path = "/content/drive/MyDrive/**/*best_model.h5"
found_models = glob.glob(search_path, recursive=True)

if len(found_models) == 0:
    raise FileNotFoundError(
        "❌ Could not find best_model.h5 in your Google Drive.\n"
        "Place the model anywhere inside MyDrive and run again."
    )

BEST_PATH = found_models[0]
print("✅ Found model at:", BEST_PATH)

# Load model
try:
    _model = load_model(BEST_PATH, compile=False)
    print("🔥 Model loaded successfully!")
except Exception as e:
    raise RuntimeError(f"❌ Failed to load model: {e}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Found model at: /content/drive/MyDrive/best_model.h5
🔥 Model loaded successfully!


In [11]:
import os, io
from pathlib import Path
from flask import Flask, request, jsonify
from werkzeug.utils import secure_filename
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import librosa
from scipy.ndimage import zoom

# === EXACT TRAINING HYPERPARAMETERS FROM YOUR NOTEBOOK ===
HYPERPARAMS = {
    "sr": 16000,
    "duration": 2.0,
    "n_fft": 2048,
    "hop_length": 512,
    "n_mels": 128,
    "decision_threshold": 0.5
}

print("Loaded Hyperparameters:", HYPERPARAMS)


Loaded Hyperparameters: {'sr': 16000, 'duration': 2.0, 'n_fft': 2048, 'hop_length': 512, 'n_mels': 128, 'decision_threshold': 0.5}


In [12]:
def preprocess_audio_for_inference_from_path(
    file_path,
    sample_rate=16000,
    duration=2.0,
    n_fft=2048,
    hop_length=512,
    n_mels=128
):
    target_length = int(sample_rate * duration)

    try:
        audio, sr = sf.read(file_path)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != sample_rate:
            audio = librosa.resample(audio.astype(np.float32), sr, sample_rate)
            sr = sample_rate
    except:
        audio, sr = librosa.load(file_path, sr=sample_rate)

    audio = audio.astype(np.float32)

    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)), "constant")
    else:
        audio = audio[:target_length]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sample_rate,
        n_fft=n_fft, hop_length=hop_length,
        n_mels=n_mels, power=2.0
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    mel_resized = zoom(
        mel_db,
        (128 / mel_db.shape[0], 128 / mel_db.shape[1]),
        order=1
    )

    eps = 1e-8
    c0 = (mel_resized - mel_resized.min()) / (mel_resized.max() - mel_resized.min() + eps)

    delta = librosa.feature.delta(mel_db)
    delta2 = librosa.feature.delta(mel_db, order=2)

    delta_resized = zoom(delta, (128/delta.shape[0], 128/delta.shape[1]), order=1)
    delta2_resized = zoom(delta2, (128/delta2.shape[0], 128/delta2.shape[1]), order=1)

    c1 = (delta_resized - delta_resized.min()) / (delta_resized.max() - delta_resized.min() + eps)
    c2 = (delta2_resized - delta2_resized.min()) / (delta2_resized.max() - delta2_resized.min() + eps)

    mel_3ch = np.stack([c0, c1, c2], axis=-1).astype(np.float32)

    return mel_3ch[np.newaxis, ...]  # (1,128,128,3)


In [13]:
def audio_bytes_to_2s_wav(bytes_in, output_path, target_sr=16000):

    audio = AudioSegment.from_file(io.BytesIO(bytes_in))
    audio = audio.set_frame_rate(target_sr).set_channels(1).set_sample_width(2)

    temp_wav = output_path
    audio.export(temp_wav, format="wav")

    data, sr = sf.read(temp_wav)
    if data.ndim > 1:
        data = data.mean(axis=1)

    target_len = int(2 * sr)
    if len(data) >= target_len:
        clip = data[:target_len]
    else:
        clip = np.pad(data, (0, target_len - len(data)), "constant")

    out_path = temp_wav.replace(".wav", "_2s.wav")
    sf.write(out_path, clip, sr)

    return out_path



In [14]:
def infer_verdict(clip_path):
    X = preprocess_audio_for_inference_from_path(
        clip_path,
        sample_rate=HYPERPARAMS["sr"],
        duration=HYPERPARAMS["duration"],
        n_fft=HYPERPARAMS["n_fft"],
        hop_length=HYPERPARAMS["hop_length"],
        n_mels=HYPERPARAMS["n_mels"]
    )

    prob = float(_model.predict(X)[0][0])

    verdict = "REAL" if prob >= HYPERPARAMS["decision_threshold"] else "FAKE"

    return {
        "verdict": verdict,
        "confidence": prob,
        "threshold": HYPERPARAMS["decision_threshold"]
    }


In [15]:
from flask_ngrok import run_with_ngrok

app = Flask(__name__)
run_with_ngrok(app)

UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(exist_ok=True)

@app.route("/analyze", methods=["POST"])
def analyze():
    if "audio" not in request.files:
        return jsonify({"error": "No audio file provided"}), 400

    file = request.files["audio"]
    raw = file.read()
    fname = secure_filename(file.filename or "audio.wav")
    temp_path = str(UPLOAD_DIR / fname)

    try:
        clip_path = audio_bytes_to_2s_wav(raw, temp_path, 16000)
    except Exception as e:
        return jsonify({"error": f"Conversion failed: {e}"}), 500

    try:
        result = infer_verdict(clip_path)
    except Exception as e:
        return jsonify({"error": f"Inference failed: {e}"}), 500

    return jsonify(result)

app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
Exception in thread Thread-4:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
            

In [18]:
from google.colab import files
test = files.upload()

fname = list(test.keys())[0]
print("Testing:", fname)

clip = audio_bytes_to_2s_wav(open(fname, "rb").read(), "/content/test.wav")
infer_verdict(clip)


Saving file2.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav to file2.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec (3).wav
Testing: file2.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec (3).wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step


{'verdict': 'REAL', 'confidence': 0.9998561143875122, 'threshold': 0.5}